In [1]:
"""
PIPELINE STAGE: Integrated Capacity–Generation Dataset Construction (Melt, Merge & Wide-Format Transformation)

INPUT:
    processed_data/steps/03_Capacity_Enriched.xlsx
    processed_data/steps/06_Generation_Enriched.xlsx

OUTPUT:
    processed_data/steps/07_Capacity_Generation.xlsx

LIBRARIES: pandas, os

1. OBJECTIVE:
    Integrate installed capacity and electricity generation datasets into a unified
    analytical structure. The pipeline reshapes both datasets into long format,
    performs a full outer merge on shared identifiers, standardizes numerical values,
    and transforms the data into a wide-format panel suitable for comparative energy
    analysis, machine learning, and econometric modeling.

2. DATA INGESTION:
    A. Capacity Dataset:
       Load enriched installed capacity data containing:
           - Province (İLLER)
           - Year (Yıl)
           - Month (Ay)
           - Plate code (Plaka)
           - Energy source columns (wide format)

    B. Generation Dataset:
       Load enriched electricity generation data with identical structure.

    C. Logging:
       Print file names upon successful loading for traceability.

3. LONG-FORM TRANSFORMATION (MELT PROCESS):
    A. Capacity Dataset Reshaping:
       Convert wide-format capacity columns into long format using:

           id_vars = ['İLLER', 'Yıl', 'Ay', 'Plaka']
           var_name = 'Kaynak'
           value_name = 'Kurulu_Güç_MW'

    B. Generation Dataset Reshaping:
       Apply identical transformation:

           value_name = 'Üretim_MWh'

    C. Resulting Structure:
       Both datasets are converted into a unified long-format schema:

           Province | Year | Month | Plate | Energy_Source | Value

4. DATA MERGING (FULL OUTER JOIN):
    A. Join Strategy:
       Merge capacity and generation datasets using:

           ['İLLER', 'Yıl', 'Ay', 'Plaka', 'Kaynak']

       with:

           how = "outer"

    B. Purpose:
       Preserve all observations from both datasets, ensuring no loss
       of capacity or generation records.

    C. Missing Value Handling:
       Replace missing numeric values with zero:

           NaN → 0

5. NUMERIC DATA NORMALIZATION:
    A. Turkish Number Format Handling:
       Convert localized numeric strings into float format.

    B. Conversion Rules:
       - Remove thousand separators (.)
       - Replace decimal commas (,) with dots (.)
       - Convert final values to float
       - Convert NaN values to 0.0

    C. Application:
       Apply conversion to:

           Kurulu_Güç_MW
           Üretim_MWh

6. WIDE-FORMAT TRANSFORMATION (PIVOT OPERATION):
    A. Pivot Structure:
       Transform long-format data into wide format using:

           index:
               ['Plaka', 'İLLER', 'Yıl', 'Ay']

           columns:
               'Kaynak'

           values:
               ['Kurulu_Güç_MW', 'Üretim_MWh']

           aggregation:
               sum

           missing values:
               fill_value = 0

    B. Result:
       Multi-index column structure:

           (Metric, Energy_Source)

7. COLUMN FLATTENING & STANDARDIZATION:
    A. MultiIndex Handling:
       Flatten pivoted columns into a single-level schema.

    B. Naming Convention:
       Generate standardized feature names:

           Capacity:
               capacity_<Energy_Source>

           Generation:
               generation_<Energy_Source>

    C. Final Structure:
       Each energy source appears twice:

           - Installed capacity (MW)
           - Electricity generation (MWh)

8. DATASET RESTRUCTURING:
    A. Index Reset:
       Convert index fields into standard columns:

           Plaka, İLLER, Yıl, Ay

    B. Column Organization:
       Final ordering strategy:

           1. Spatial identifiers (Plaka, İLLER)
           2. Capacity features (sorted)
           3. Temporal features (Yıl, Ay)
           4. Generation features (sorted)

9. DATA INTEGRITY & ALIGNMENT:
    A. Consistency Assurance:
       Ensure that capacity and generation are aligned by:
           - Province
           - Year
           - Month
           - Energy source

    B. Missing Data Strategy:
       Replace all missing values with zero after integration.

10. OUTPUT GENERATION:
    A. Final Dataset Export:

           processed_data/steps/07_Capacity_Generation.xlsx

    B. Export Settings:
       - Excel format (.xlsx)
       - No index column

11. PROCESS LOGGING & MONITORING:
    A. Runtime Messages:
       Display:
           - Capacity dataset loading confirmation
           - Generation dataset loading confirmation
           - Merge completion message
           - Pivot transformation status
           - Final save confirmation

    B. Warning Considerations:
       - Missing energy source mismatches
       - Inconsistent province naming
       - Unaligned temporal records

12. EXPECTED OUTPUT DATASET:
    The resulting dataset contains:

        - Province-level energy system panel data
        - Installed capacity by energy source (MW)
        - Electricity generation by energy source (MWh)
        - Temporal dimensions (Year, Month)
        - Spatial identifier (Plaka, Province)

    The dataset is structured in wide format and is suitable for:

        - Energy transition modeling
        - Capacity vs generation efficiency analysis
        - Panel regression models
        - Machine learning forecasting
        - Spatial-temporal energy system studies
"""

import pandas as pd
import os

base_dir = r"C:\Users\w11\dergi2"
cap_path = os.path.join(base_dir, "processed_data", "steps", "03_capacity_enriched.xlsx")
gen_path = os.path.join(base_dir, "processed_data", "steps", "06_generation_enriched.xlsx")
output_path = os.path.join(base_dir, "processed_data", "steps", "07_capacity_generation.xlsx")

df_cap = pd.read_excel(cap_path)
print(f">>> Kapasite Verisi okunuyor: {os.path.basename(cap_path)}")

df_gen = pd.read_excel(gen_path)
print(f">>> Üretim Verisi okunuyor: {os.path.basename(gen_path)}")

# 1. Kurulu Güç Melt
cap_melted = df_cap.melt(id_vars=['İLLER', 'Yıl', 'Ay', 'Plaka'], 
                         var_name='Kaynak', 
                         value_name='Kurulu_Güç_MW')

# 2. Üretim Melt
gen_melted = df_gen.melt(id_vars=['İLLER', 'Yıl', 'Ay', 'Plaka'], 
                         var_name='Kaynak', 
                         value_name='Üretim_MWh')

# 3. Birleştirme (Tam eşleşme)
master_df = pd.merge(cap_melted, gen_melted, 
                     on=['İLLER', 'Yıl', 'Ay', 'Plaka', 'Kaynak'], 
                     how='outer')
print(">>> Tablolar birleştirildi, yan yana formata (pivot) geçiliyor...")

# NaN değerleri 0 ile doldur
master_df['Kurulu_Güç_MW'] = master_df['Kurulu_Güç_MW'].fillna(0)
master_df['Üretim_MWh'] = master_df['Üretim_MWh'].fillna(0)

# --- SAYISAL DÖNÜŞÜM BLOĞU (PIVOT'TAN ÖNCE) ---

def turkce_sayiyi_floata_cevir(deger):
    """
    11.724,59 gibi string formatındaki Türkçe sayıları 
    11724.59 şeklinde float (ondalık sayı) formatına çevirir.
    """
    if pd.isna(deger): # Eğer değer boşsa (NaN), 0 döndür
        return 0.0
    if isinstance(deger, str):
        # 1. Adım: Binlik ayracı olan noktaları tamamen sil
        deger = deger.replace('.', '')
        # 2. Adım: Ondalık ayracı olan virgülü noktaya çevir
        deger = deger.replace(',', '.')
    
    return float(deger)

print(">>> String değerler sayısal formatlara (float) dönüştürülüyor...")

# Dönüşüm fonksiyonunu Kurulu Güç ve Üretim sütunlarına uygula
master_df['Kurulu_Güç_MW'] = master_df['Kurulu_Güç_MW'].apply(turkce_sayiyi_floata_cevir)
master_df['Üretim_MWh'] = master_df['Üretim_MWh'].apply(turkce_sayiyi_floata_cevir)

# ----------------------------------------------


# 4. Pivot İşlemi (Geniş Formata Çevirme)
wide_df = master_df.pivot_table(
    index=['Plaka', 'İLLER', 'Yıl', 'Ay'],
    columns='Kaynak',
    values=['Kurulu_Güç_MW', 'Üretim_MWh'],
    aggfunc='sum',
    fill_value=0
)

# 5. Sütun İsimlerini Düzenleme (MultiIndex Düzleştirme)
# wide_df.columns şu an iki katmanlı: ('Kurulu_Güç_MW', 'Biyokütle') vb.
yeni_sutunlar = []
for metrik, kaynak in wide_df.columns:
    if metrik == 'Kurulu_Güç_MW':
        yeni_sutunlar.append(f"capacity_{kaynak}")
    else: # Üretim_MWh
        yeni_sutunlar.append(f"generation_{kaynak}")

wide_df.columns = yeni_sutunlar

# İndeksi sıfırlayarak Plaka, İLLER, Yıl ve Ay'ı normal sütunlara dönüştür
wide_df = wide_df.reset_index()

# 6. Sütun Sıralamasını İstenen Formata Getirme
# capacity ve generation sütunlarını ayıklayıp listeliyoruz
sabit_sutunlar_bas = ['Plaka', 'İLLER']
capacity_sutunlari = sorted([col for col in wide_df.columns if col.startswith('capacity_')])
zaman_sutunlari = ['Yıl', 'Ay']
generation_sutunlari = sorted([col for col in wide_df.columns if col.startswith('generation_')])

# Sütunları tam istediğin sırayla birleştir
nihai_sira = sabit_sutunlar_bas + capacity_sutunlari + zaman_sutunlari + generation_sutunlari
wide_df = wide_df[nihai_sira]

# 7. Kayıt
wide_df.to_excel(output_path, index=False)
print(">>> İşlem tamamlandı, tablo yan yana formatta hazırlandı.")

>>> Kapasite Verisi okunuyor: 03_capacity_enriched.xlsx
>>> Üretim Verisi okunuyor: 06_generation_enriched.xlsx
>>> Tablolar birleştirildi, yan yana formata (pivot) geçiliyor...
>>> String değerler sayısal formatlara (float) dönüştürülüyor...
>>> İşlem tamamlandı, tablo yan yana formatta hazırlandı.


In [3]:
import pandas as pd
import os

base_dir = r"C:\Users\w11\dergi2"
cap_path = os.path.join(base_dir, "processed_data", "steps", "03_capacity_enriched.xlsx")
gen_path = os.path.join(base_dir, "processed_data", "steps", "06_generation_enriched.xlsx")

output_1 = os.path.join(base_dir, "processed_data", "steps", "07_capacity_generation.xlsx")
output_2 = os.path.join(base_dir, "processed_data", "steps", "07_generation_capacity.xlsx")

df_cap = pd.read_excel(cap_path)
df_gen = pd.read_excel(gen_path)

# 1. Melt (Düzleştirme)
cap_melted = df_cap.melt(id_vars=['İLLER', 'Yıl', 'Ay', 'Plaka'], var_name='Kaynak', value_name='Kurulu_Güç_MW')
gen_melted = df_gen.melt(id_vars=['İLLER', 'Yıl', 'Ay', 'Plaka'], var_name='Kaynak', value_name='Üretim_MWh')

def isleyip_kaydet(sol_df, sag_df, output_file):
    # Left Join: Ana tablo sol_df baz alınır, satır sayısı sol_df'in benzersiz kayıtlarına eşitlenir
    master_df = pd.merge(sol_df, sag_df, on=['İLLER', 'Yıl', 'Ay', 'Plaka', 'Kaynak'], how='left')
    master_df = master_df.fillna(0)

    # Sayısal Dönüşüm (String'den Float'a)
    for col in ['Kurulu_Güç_MW', 'Üretim_MWh']:
        if col in master_df.columns:
            master_df[col] = master_df[col].astype(str).str.replace('.', '', regex=False)
            master_df[col] = master_df[col].str.replace(',', '.', regex=False).astype(float)

    # Pivot (Yan Yana Formata Çevirme)
    wide_df = master_df.pivot_table(index=['Plaka', 'İLLER', 'Yıl', 'Ay'], columns='Kaynak', 
                                    values=['Kurulu_Güç_MW', 'Üretim_MWh'], aggfunc='sum', fill_value=0)
    
    # Sütun İsimlerini Düzenleme
    yeni_sutunlar = [f"capacity_{k}" if m == 'Kurulu_Güç_MW' else f"generation_{k}" for m, k in wide_df.columns]
    wide_df.columns = yeni_sutunlar
    wide_df = wide_df.reset_index()

    # İstenen Sıralama
    sabitler = ['Plaka', 'İLLER']
    zaman = ['Yıl', 'Ay']
    cap_sutunlar = sorted([c for c in wide_df.columns if c.startswith('capacity_')])
    gen_sutunlar = sorted([c for c in wide_df.columns if c.startswith('generation_')])
    
    wide_df = wide_df[sabitler + cap_sutunlar + zaman + gen_sutunlar]

    # Kaydet
    wide_df.to_excel(output_file, index=False)
    print(f">>> Kaydedildi: {os.path.basename(output_file)} | Satır Sayısı: {len(wide_df)}")

# 1. Senaryo: Kapasite Tablosu Baz Alınarak (Hedef: 4791)
isleyip_kaydet(cap_melted, gen_melted, output_1)

# 2. Senaryo: Üretim Tablosu Baz Alınarak (Hedef: 4671/4672)
isleyip_kaydet(gen_melted, cap_melted, output_2)

>>> Kaydedildi: 07_capacity_generation.xlsx | Satır Sayısı: 4790
>>> Kaydedildi: 07_generation_capacity.xlsx | Satır Sayısı: 4827
